In [1]:
import requests

In [3]:
# Connection strings
CLIENT_ID = "3t6hp09w7jck16tyvkho4p2rgtd9f3"
CLIENT_SECRET = "i6vg2u4i4v631fo3nfdrui5wdxyvx7"

In [4]:
# Getting OAuth token
def get_app_token():
    url = "https://id.twitch.tv/oauth2/token"
    params = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials"
             }
    response = requests.post(url, data=params)
    response.raise_for_status()
    return response.json()["access_token"]    
access_token = get_app_token()
print("Access token :", access_token)

Access token : r2ra909mnvban6nqm02klmepm2pz1a


In [9]:
CHANNEL_LOGIN = "esl_dota2"
OUTPUT_CSV = "esl_dota2_vods.csv"

In [5]:
# --- Step 2: Get User ID ---
def get_user_id(access_token):
    headers = {
        "Client-ID": CLIENT_ID,
        "Authorization": f"Bearer {access_token}"
    }
    response = requests.get(
        "https://api.twitch.tv/helix/users",
        headers=headers,
        params={"login": CHANNEL_LOGIN}
    )
    response.raise_for_status()
    return response.json()["data"][0]["id"]

In [6]:
# --- Step 3: Fetch All Archived VODs ---
def get_all_vods(user_id, access_token):
    vods = []
    url = "https://api.twitch.tv/helix/videos"
    headers = {
        "Client-ID": CLIENT_ID,
        "Authorization": f"Bearer {access_token}"
    }
    params = {
        "user_id": user_id,
        "type": "archive",
        "first": 100  # max per page
    }

    while True:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        vods.extend(data["data"])

        if "pagination" in data and "cursor" in data["pagination"]:
            params["after"] = data["pagination"]["cursor"]
        else:
            break

    return vods

In [7]:
# --- Step 4: Save to CSV ---
def save_vods_to_csv(vods):
    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=["id", "title", "created_at", "duration", "url"])
        writer.writeheader()
        for v in vods:
            writer.writerow({
                "id": v["id"],
                "title": v["title"],
                "created_at": v["created_at"],
                "duration": v["duration"],
                "url": v["url"]
            })

In [11]:
import csv


In [12]:
# --- MAIN EXECUTION ---
if __name__ == "__main__":
    print("🔐 Getting OAuth token...")
    token = get_app_token()

    print("🎯 Fetching user ID for channel:", CHANNEL_LOGIN)
    user_id = get_user_id(token)

    print("📺 Fetching all archived VODs...")
    vods = get_all_vods(user_id, token)
    print(f"✅ Found {len(vods)} VODs")

    print(f"💾 Saving to CSV: {OUTPUT_CSV}")
    save_vods_to_csv(vods)
    print("🎉 Done!")

🔐 Getting OAuth token...
🎯 Fetching user ID for channel: esl_dota2
📺 Fetching all archived VODs...
✅ Found 2708 VODs
💾 Saving to CSV: esl_dota2_vods.csv
🎉 Done!


In [13]:
import pandas as pd

In [14]:
df = pd.read_csv("esl_dota2_vods.csv")

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2708 entries, 0 to 2707
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          2708 non-null   int64 
 1   title       2708 non-null   object
 2   created_at  2708 non-null   object
 3   duration    2708 non-null   object
 4   url         2708 non-null   object
dtypes: int64(1), object(4)
memory usage: 105.9+ KB


In [18]:
df

,id,title,created_at,duration,url
0,2483328608,RERUN: Nigma Galaxy vs PARIVISION - DreamLeagu...,2025-06-11T18:41:13Z,22h2m50s,https://www.twitch.tv/videos/2483328608
1,2481582342,RERUN: Shopify Rebellion vs BetBoom Team - Dre...,2025-06-09T18:40:59Z,47h59m57s,https://www.twitch.tv/videos/2481582342
2,2479788818,RERUN: Aurora vs Flipster Talon - DreamLeague ...,2025-06-07T18:40:41Z,47h59m57s,https://www.twitch.tv/videos/2479788818
3,2477959972,RERUN: Xtreme Gaming vs BOOM Esports - DreamLe...,2025-06-05T18:40:25Z,47h59m57s,https://www.twitch.tv/videos/2477959972
4,2476200706,RERUN: Flipster Talon vs Gaimin Gladiators - D...,2025-06-03T18:40:17Z,47h59m57s,https://www.twitch.tv/videos/2476200706
...,...,...,...,...,...
2703,38008216,rat in the dark vs. Absolute Legends - Quarter...,2013-04-20T11:15:38Z,2m36s,https://www.twitch.tv/videos/38008216
2704,38008252,rat in the dark vs. Absolute Legends - Quarter...,2013-04-20T09:29:10Z,2h11m37s,https://www.twitch.tv/videos/38008252
2705,38008234,Raidcall EMS One Finals Drawing,2013-04-20T08:23:03Z,31m40s,https://www.twitch.tv/videos/38008234
2706,38008350,Raidcall EMS One Finals Drawing,2013-04-05T15:09:29Z,9m54s,https://www.twitch.tv/videos/38008350
